In [ ]:
# Cargamos las librerías necesarias
import matplotlib.pyplot as plt
import numpy as np
import time
import networkx as nx
from collections import deque

In [ ]:
# Aqui se definen todas las funciones necesarias para resolver un CSP utilizando AC-3 y backtracking.

# Contador global para nodos visitados
node_count = 0

def draw_graph(variables, domains, neighbors, title=""):
    '''
    Dibuja el grafo de un CSP usando NetworkX y Matplotlib.
    - variables: lista de variables del CSP.
    - domains: diccionario que asigna a cada variable su dominio actual.
    - neighbors: diccionario que asigna a cada variable su lista de variables vecinas.
    - title: título opcional para el gráfico.
    '''
    G = nx.Graph()

    for var in variables:
        G.add_node(var)

    for xi in neighbors:
        for xj in neighbors[xi]:
            if not G.has_edge(xi, xj):
                G.add_edge(xi, xj)

    pos = nx.circular_layout(G)

    labels = {v: f"{v}\n{domains[v]}" for v in variables}

    colors = []
    for v in variables:
        if len(domains[v]) == 0:
            colors.append("red")       # inconsistente
        elif len(domains[v]) == 1:
            colors.append("lightgreen") # casi asignado
        else:
            colors.append("lightblue")  # normal

    plt.figure(figsize=(6,6))
    nx.draw(G, pos, with_labels=True, labels=labels,
            node_color=colors, node_size=2500,
            font_size=10)

    plt.title(title)
    plt.show()

def constraint(xi, vi, xj, vj):
    return True  # Función de restricción auxiliar que siempre devuelve True (sin restricciones)

def revise(domains, xi, xj):
    '''
    Revisa el dominio de xi para asegurarse de que cada valor tenga al menos un valor compatible en xj.
    - domains: diccionario de dominios actuales.
    - xi: variable a revisar.
    - xj: variable vecina que se utiliza para revisar xi.
    Devuelve True si se revisó el dominio de xi, False si no se hizo ningún cambio.
    '''
    revised = False
    to_remove = []

    for vi in domains[xi]:
        if not any(constraint(xi, vi, xj, vj) for vj in domains[xj]):
            to_remove.append(vi)
            print(f"Revise: Removing {vi} from {xi} because no value in {xj} satisfies the constraint with {vi}")
            
    for vi in to_remove:
        domains[xi].remove(vi)
        revised = True
        
    return revised

def ac3(domains, neighbors):
    '''   
    Aplica el algoritmo AC-3 para hacer que el CSP sea arc-consistente.
    - domains: diccionario de dominios actuales.
    - neighbors: diccionario que asigna a cada variable su lista de variables vecinas.
    Devuelve False si se encuentra un dominio vacío (inconsistente), None si se completa sin inconsistencias.
    '''
    queue = deque()

    # Inicializar cola con todos los arcos
    for xi in neighbors:
        for xj in neighbors[xi]:
            queue.append((xi, xj))

    step = 0

    while queue:
        xi, xj = queue.popleft()
        step += 1

        print(f"\nPaso {step}: Revisando arco ({xi}, {xj})")
        print(f"Dominios antes: {xi}={domains[xi]}, {xj}={domains[xj]}")

        if revise(domains, xi, xj):
            print(f"⚠️ Cambio en {xi} → nuevo dominio: {domains[xi]}")

            if len(domains[xi]) == 0:
                print(f"❌ Dominio vacío en {xi}. CSP inconsistente.")
                return False

            for xk in neighbors[xi]:
                if xk != xj:
                    queue.append((xk, xi))
                    print(f"   ↪ Se agrega ({xk}, {xi}) a la cola")
        else:
            print("Sin cambios")

    return None

def is_valid(assignment, var, value, neighbors, constraint):
    '''
    Verifica si un valor es válido para una variable dada en la asignación actual.
    - assignment: diccionario que asigna a cada variable su valor asignado.
    - var: variable para la cual se verifica la validez del valor.
    - value: valor a verificar.
    - neighbors: diccionario que asigna a cada variable su lista de variables vecinas.
    - constraint: función de restricción que toma dos variables y sus valores y devuelve True si la restricción se cumple.
    Devuelve True si el valor es válido, False en caso contrario.
    '''
    for neighbor in neighbors[var]:
        if neighbor in assignment and not constraint(var, value, neighbor, assignment[neighbor]):
            return False
    return True

def backtracking_search(variables, domains, neighbors, constraint):
    '''
        Función principal de búsqueda con backtracking para resolver el CSP.
        - variables: lista de variables del CSP.
        - domains: diccionario que asigna a cada variable su dominio actual.
        - neighbors: diccionario que asigna a cada variable su lista de variables vecinas.
        - constraint: función de restricción que toma dos variables y sus valores y devuelve True si la restricción se cumple.
        Devuelve una asignación completa si se encuentra una solución, None en caso contrario.
        Se utiliza una función recursiva interna 'backtrack' para construir la asignación paso a paso. 
    '''
    def backtrack(assignment):
        ''' 
        Función recursiva de backtracking que intenta construir una asignación completa.
        - assignment: diccionario que asigna a cada variable su valor asignado hasta el momento.
        Devuelve una asignación completa si se encuentra una solución, None en caso contrario.
        '''
        global node_count
        node_count += 1
    
        if len(assignment) == len(variables):
            return assignment

        unassigned = [v for v in variables if v not in assignment]
        
        var = unassigned[0]
        for value in domains[var]:
            if is_valid(assignment, var, value, neighbors, constraint):
                assignment[var] = value
                result = backtrack(assignment)
                if result:
                    return result
                del assignment[var]

        return None

    return backtrack({})

def forward_checking_search(variables, domains, neighbors, constraint):
    ''' Función de búsqueda con forward checking para resolver el CSP.
    - variables: lista de variables del CSP.
    - domains: diccionario que asigna a cada variable su dominio actual.
    - neighbors: diccionario que asigna a cada variable su lista de variables vecinas.
    - constraint: función de restricción que toma dos variables y sus valores y devuelve True si la restricción se cumple.
    Devuelve una asignación completa si se encuentra una solución, None en caso contrario.
    Se utiliza una función recursiva interna 'forward_checking' que intenta construir una asignación completa 
    mientras realiza forward checking para reducir los dominios de las variables vecinas.
    '''
    def forward_checking(assignment):
        ''' Función recursiva de forward checking que intenta construir una asignación completa.
        - assignment: diccionario que asigna a cada variable su valor asignado hasta el momento.
        Devuelve una asignación completa si se encuentra una solución, None en caso contrario.
        '''
        global node_count
        node_count += 1

        if len(assignment) == len(variables):
            return assignment

        var = [v for v in domains if not v in assignment][0]

        for value in domains[var]:
            if is_valid(assignment, var, value, neighbors, constraint):
                local_domains = {v:list(domains[v]) for v in domains}
                assignment[var] = value

                for y in neighbors[var]:
                    if y not in assignment:
                        local_domains[y] = [v for v in local_domains[y] if v != value]

                result = forward_checking(assignment)

                if result:
                    return result

                del assignment[var]
        return None
    
    return forward_checking({})

# Problema de las 4 reinas

La meta del problema de las 4 reinas es poner 4 reinas en un tablero de ajedrez de 4x4, de manera que ninguna de ellas esté en posibilidad de atacarse mutuamente (no esté en la misma fila, o columna o diagonal)

In [ ]:
# Visualizar el tablero
N = 4
board = np.add.outer(range(N), range(N)) % 2

for col in range(N):
    plt.text(col, -1, '♛', fontsize=30, ha='center', va='center', color='red')

plt.imshow(board, cmap='binary_r')
plt.axis('on')  # Hide axis ticks
plt.xticks(range(N))
plt.yticks(range(N))
plt.show()

In [ ]:
# Aqui se define el CSP para el problema de las 4 reinas (variables, dominios, restricciones y vecinos)

# Cada variable representa la fila donde se coloca la reina en la columna correspondiente (x1 para columna 1, etc.)
variables = ["x1", "x2", "x3", "x4"] 

# Dominio de cada variable: las filas posibles para colocar la reina en cada columna
domains = {
    "x1": [1,2,3,4],
    "x2": [1,2,3,4],
    "x3": [1,2,3,4],
    "x4": [1,2,3,4]
}

# Restricciones: no pueden estar en la misma fila, ni en la misma diagonal
def constraint(xi, vi, xj, vj):

    if xi != xj and vi == vj:
        return False
    
    if (xi, xj) in [("x1", "x2"), ("x2", "x1"), ("x2", "x3"), ("x3", "x2"), ("x3", "x4"), ("x4", "x3")]:
        return abs(vi - vj) != 1
    
    if (xi, xj) in [("x1", "x3"), ("x3", "x1"), ("x2", "x4"), ("x4", "x2")]: 
        return abs(vi - vj) != 2
    
    if (xi, xj) in [("x1", "x4"), ("x4", "x1")]: 
        return abs(vi - vj) != 3

    return True

# Vecinos: cada variable es vecina de las otras 3 (todas las variables están conectadas entre sí)
neighbors = {
    'x1': ['x2', 'x3', 'x4'],
    'x2': ['x1', 'x3', 'x4'],
    'x3': ['x1', 'x2', 'x4'],
    'x4': ['x1', 'x2', 'x3']
}

In [ ]:
# Visualizar el grafo de restricciones
draw_graph(variables, domains, neighbors, title="Grafo de restricciones")

In [ ]:
# Aplicar AC-3 para hacer el CSP arc-consistente
ac3(domains, neighbors)

In [ ]:
global node_count
node_count = 0

start_time = time.time() # Medir el tiempo de ejecución comienzo   

sol = backtracking_search(variables, domains, neighbors, constraint) # Ejecutar búsqueda con backtracking

end_time = time.time() # Medir el tiempo de ejecución final

print(f"\nSolución encontrada: {sol}")
print(f"Nodos explorados: {node_count}")
print(f"Tiempo de ejecución: {end_time - start_time:.2f} segundos")

In [ ]:
# Visualizar la solución en el tablero
board = np.add.outer(range(N), range(N)) % 2

plt.imshow(board, cmap='binary_r')

for var, value in sol.items():
    col = int(var[1]) - 1
    row = value - 1
    plt.text(col, row, '♛', fontsize=30, ha='center', va='center', color='red')

plt.axis('on')
plt.xticks(range(N))
plt.yticks(range(N))
plt.show()

## Problema de N-reinas

In [ ]:
N = 5
variables = [f'queen_{i}' for i in range(1, N+1)]

domains = {var: list(range(1, N+1)) for var in variables}

def constraint(xi, vi, xj, vj):
    if xi != xj and vi == vj:
        return False
    if abs(int(xi.split('_')[1]) - int(xj.split('_')[1])) == abs(vi - vj):
        return False
    return True

neighbors = {var: [v for v in variables if v != var] for var in variables}

In [ ]:
draw_graph(variables, domains, neighbors, title="Grafo de restricciones")

In [ ]:
ac3(domains, neighbors)

In [ ]:
global node_count
node_count = 0
start_time = time.time()
sol = backtracking_search(variables, domains, neighbors, constraint)
end_time = time.time()  
print(f"\nSolución encontrada: {sol}")
print(f"Nodos explorados: {node_count}")
print(f"Tiempo de ejecución: {end_time - start_time} segundos")

In [ ]:
board = np.add.outer(range(N), range(N)) % 2

for var, value in sol.items():
    col = int(var.split('_')[1]) - 1
    row = value - 1
    plt.text(col, row, '♛', fontsize=30, ha='center', va='center', color='red')

plt.imshow(board, cmap='binary_r')
plt.axis('on')  # Hide axis ticks
plt.xticks(range(N))
plt.yticks(range(N))
plt.show()

### Forward checking

In [ ]:
N = 10
variables = [f'queen_{i}' for i in range(1, N+1)]

domains = {var: list(range(1, N+1)) for var in variables}

def constraint(xi, vi, xj, vj):
    if xi != xj and vi == vj:
        return False
    if abs(int(xi.split('_')[1]) - int(xj.split('_')[1])) == abs(vi - vj):
        return False
    return True

neighbors = {var: [v for v in variables if v != var] for var in variables}

In [ ]:
global node_count
node_count = 0
start_time = time.time()
sol = forward_checking_search(variables, domains, neighbors, constraint)
end_time = time.time()
print(f"\nSolución encontrada: {sol}")
print(f"Nodos explorados: {node_count}")
print(f"Tiempo de ejecución: {end_time - start_time:.2f} segundos")

# Cuadrado Latino

In [ ]:
N = 3
letters = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"

variables = [f"x_{i}{j}" for i in range(N) for j in range(N)]

domains = { x: [letter for letter in letters[:N]] for x in variables }

def constraint(xi, vi, xj, vj):

    if xi not in domains or xj not in domains:
        return False 
    
    col_i, row_i = int(xi[2]), int(xi[3])
    col_j, row_j = int(xj[2]), int(xj[3])

    if col_i == col_j:
        return vi != vj
    if row_i == row_j:
        return vi != vj
 
    return True

neighbors = { 
    x: [f"x_{i}{j}" for i in range(N) for j in range(N) if (f"x_{i}{j}" != x) and (int(x[2]) == i or int(x[3]) == j)] for x in variables
}

In [ ]:
draw_graph(variables, domains, neighbors, title="CSP Inicial")

In [ ]:
ac3(domains, neighbors)

In [ ]:
global node_count
node_count = 0
sol = backtracking_search(variables, domains, neighbors, constraint)
print(f"\nSolución encontrada: {sol}")
print(f"Nodos explorados: {node_count}")

In [ ]:
board = np.add.outer(range(N), range(N)) % 2

for var, value in sol.items():
    col = int(var[3])
    row = int(var[2])
    plt.text(col, row, value, fontsize=30, ha='center', va='center', color='red')

plt.imshow(board, cmap='binary_r')
plt.axis('on')
plt.xticks(range(N))
plt.yticks(range(N))
plt.show()

# Sudoku

In [ ]:
N = 9

variables = [f"x_{i}{j}" for i in range(N) for j in range(N)]

domains = { x: [i for i in range(1,N+1)] for x in variables }

domains["x_02"] = [3]
domains["x_04"] = [2]
domains["x_06"] = [6]
domains["x_10"] = [9]
domains["x_13"] = [3]
domains["x_15"] = [5]
domains["x_18"] = [1]
domains["x_22"] = [1]
domains["x_23"] = [8]
domains["x_25"] = [6]
domains["x_26"] = [4]
domains["x_32"] = [8]
domains["x_33"] = [1]
domains["x_35"] = [2]
domains["x_36"] = [9]
domains["x_40"] = [7]
domains["x_48"] = [8]
domains["x_52"] = [6]
domains["x_53"] = [7]
domains["x_55"] = [8]
domains["x_56"] = [2]
domains["x_62"] = [2]
domains["x_63"] = [6]
domains["x_65"] = [9]
domains["x_66"] = [5]
domains["x_70"] = [8]
domains["x_73"] = [2]
domains["x_75"] = [3]
domains["x_78"] = [9]
domains["x_82"] = [5]
domains["x_84"] = [1]
domains["x_86"] = [3]

def constraint(xi, vi, xj, vj):

    if xi not in domains or xj not in domains:
        return False  
    
    col_i, row_i = int(xi[2]), int(xi[3])
    col_j, row_j = int(xj[2]), int(xj[3])

    if col_i == col_j:
        return vi != vj
    if row_i == row_j:
        return vi != vj
    
    block_i = (col_i // 3, row_i // 3)
    block_j = (col_j // 3, row_j // 3)
    if block_i == block_j:
        return vi != vj

    return True

neighbors = { x: [f"x_{i}{j}" for i in range(N) for j in range(N) if (f"x_{i}{j}" != x) 
                  and (int(x[2]) == i 
                       or int(x[3]) == j 
                       or ((int(x[2]) // 3, int(x[3]) // 3) == (i // 3, j // 3))) ] for x in variables }

In [ ]:
start = time.time()
ac3(domains, neighbors)
end = time.time()
print(f"Tiempo de ejecución: {end - start} segundos")

In [ ]:
global node_count
node_count = 0
sol = backtracking_search(variables, domains, neighbors, constraint)
print(f"\nSolución encontrada: {sol}")
print(f"Nodos explorados: {node_count}")

In [ ]:
board = np.add.outer(range(N), range(N)) % 2

for var, value in sol.items():
    col = int(var[3])
    row = int(var[2])
    plt.text(col, row, value, fontsize=30, ha='center', va='center', color='red')

plt.imshow(board, cmap='binary_r')
plt.axis('on')  # Hide axis ticks
plt.xticks(range(N))
plt.yticks(range(N))
plt.show()